# Look at mapping breadth and depth of 100 x metagenomes to singleclust genes

In [1]:
import polars as pl
import glob
import os

## Read in individual metag x species singleclust depth reports

These files contain depth-of-mapping information for all our genes.

In [2]:
DIR='../outputs.mapping/cds3/outputs.singleclust/'
template = '../outputs.mapping/cds3/outputs.singleclust/{metag}.x.{species}.depth.txt'

def read_depth(metag, species):
    filename = template.format(metag=metag, species=species)
    df = pl.read_csv(filename, separator='\t', has_header=False,
                 new_columns=('gene', 'pos', 'cov', 'foo'))
    sum_df = df.group_by('gene').agg(
        ((pl.col("cov") > 0).sum() / pl.col("pos").len()).alias("breadth")
    ).with_columns(
        (pl.lit(metag).alias("metag")),
        (pl.lit(species).alias("species"))
)
    return sum_df

read_depth('ERR1135199', 's__Cryptobacteroides sp900546925')

gene,breadth,metag,species
str,f64,str,str
"""GDFPIIAG_01475""",0.643875,"""ERR1135199""","""s__Cryptobacteroides sp9005469…"
"""OIGHMFGK_00345""",0.685315,"""ERR1135199""","""s__Cryptobacteroides sp9005469…"
"""NEEJCNGM_00770""",0.487443,"""ERR1135199""","""s__Cryptobacteroides sp9005469…"
"""HEEBLBEE_00151""",0.772727,"""ERR1135199""","""s__Cryptobacteroides sp9005469…"
"""FOKFIHIM_01101""",0.471735,"""ERR1135199""","""s__Cryptobacteroides sp9005469…"
…,…,…,…
"""FNIGJHLI_01271""",0.668721,"""ERR1135199""","""s__Cryptobacteroides sp9005469…"
"""OIBHKANM_00256""",0.372776,"""ERR1135199""","""s__Cryptobacteroides sp9005469…"
"""FNIGJHLI_02317""",0.603297,"""ERR1135199""","""s__Cryptobacteroides sp9005469…"


In [3]:
# Read them all in!
filenames = glob.glob(f"{DIR}/*.depth.txt")

dflist = []
for i, n in enumerate(filenames):
    if i % 100 == 0:
        print(f"{i} of {len(filenames)}")
    n = os.path.basename(n)
    metag, _, species, _ = n.split('.', 3)
    dflist.append(read_depth(metag, species))

depth_df = pl.concat(dflist)

print(f"read {len(filenames)} depth files.")

0 of 1400
100 of 1400
200 of 1400
300 of 1400
400 of 1400
500 of 1400
600 of 1400
700 of 1400
800 of 1400
900 of 1400
1000 of 1400
1100 of 1400
1200 of 1400
1300 of 1400
read 1400 depth files.


In [4]:
depth_df

gene,breadth,metag,species
str,f64,str,str
"""APOLKBKG_00160""",1.0,"""ERR8314788""","""s__Prevotella sp000434975"""
"""CJIDBKAA_00098""",1.0,"""ERR8314788""","""s__Prevotella sp000434975"""
"""IBOOIAKE_01392""",0.998325,"""ERR8314788""","""s__Prevotella sp000434975"""
"""EEMFJJMB_00060""",1.0,"""ERR8314788""","""s__Prevotella sp000434975"""
"""DEENMOAI_01503""",0.92452,"""ERR8314788""","""s__Prevotella sp000434975"""
…,…,…,…
"""DKAPMIEG_00121""",0.773083,"""SRR8960391""","""s__Mogibacterium_A kristiansen…"
"""EBLLCBKE_00022""",0.307116,"""SRR8960391""","""s__Mogibacterium_A kristiansen…"
"""PKGEEMPI_00758""",0.975387,"""SRR8960391""","""s__Mogibacterium_A kristiansen…"


## Summarize our mapping depth results across all the metagenomes

In [5]:
# require 10% of each gene to be covered by at least one read
BREADTH_CUTOFF = 0.1

In [6]:
# aggregate across all metagenomes;
# calculate fraction of metagenomes for which gene mapping exceeds our breadth cutoff
agg_df = depth_df.group_by(['species', 'gene']).agg(
    ((pl.col("breadth") >= BREADTH_CUTOFF).sum() / pl.col("breadth").len()).alias("f")
)
agg_df

species,gene,f
str,str,f64
"""s__Prevotella sp000434975""","""PMHDHLGK_00474""",0.87
"""s__Cryptobacteroides sp9005469…","""HMMHNFMB_01134""",0.72
"""s__UBA2868 sp004552595""","""JFNHJANI_00488""",0.65
"""s__Sodaliphilus sp004557565""","""PMNGKPHB_01705""",0.89
"""s__Prevotella sp000434975""","""CHAJGCIB_00125""",0.76
…,…,…
"""s__Sodaliphilus sp004557565""","""ALDHKKPD_00051""",0.95
"""s__Gemmiger qucibialis""","""AGHFCGHF_02266""",0.89
"""s__Prevotella sp002251295""","""AAJNCELP_01688""",0.62


In [10]:
# print information out by species
for species in sorted(agg_df['species'].unique()):
    print(species)
    foo_df = agg_df.filter(pl.col("species") == species)
    print(foo_df.sort(by='f').filter(pl.col('f') > 0.8))

s__Bariatricus sp004560705
shape: (22, 3)
┌────────────────────────────┬────────────────┬──────┐
│ species                    ┆ gene           ┆ f    │
│ ---                        ┆ ---            ┆ ---  │
│ str                        ┆ str            ┆ f64  │
╞════════════════════════════╪════════════════╪══════╡
│ s__Bariatricus sp004560705 ┆ ELOEBFJM_01459 ┆ 0.81 │
│ s__Bariatricus sp004560705 ┆ KJMAJKMB_02318 ┆ 0.81 │
│ s__Bariatricus sp004560705 ┆ LNGHPPMH_01876 ┆ 0.81 │
│ s__Bariatricus sp004560705 ┆ EMNHIHKA_01693 ┆ 0.82 │
│ s__Bariatricus sp004560705 ┆ LNGHPPMH_00679 ┆ 0.83 │
│ …                          ┆ …              ┆ …    │
│ s__Bariatricus sp004560705 ┆ NDCIOGGG_00214 ┆ 0.89 │
│ s__Bariatricus sp004560705 ┆ AIEIMHOC_00182 ┆ 0.89 │
│ s__Bariatricus sp004560705 ┆ LEHFNBLN_02301 ┆ 0.93 │
│ s__Bariatricus sp004560705 ┆ LEHFNBLN_00701 ┆ 0.95 │
│ s__Bariatricus sp004560705 ┆ CHOLCMOC_00863 ┆ 0.97 │
└────────────────────────────┴────────────────┴──────┘
s__Cryptobacteroides sp

## Explore specific gene/species/metag combinations

In [8]:
depth_df.filter((pl.col("gene") == 'EHOAPHDI_01174')).sort(by='breadth')

gene,breadth,metag,species
str,f64,str,str
"""EHOAPHDI_01174""",0.0,"""SRR12795775""","""s__Phascolarctobacterium_A suc…"
"""EHOAPHDI_01174""",0.0,"""SRR8655118""","""s__Phascolarctobacterium_A suc…"
"""EHOAPHDI_01174""",0.0,"""ERR3212024""","""s__Phascolarctobacterium_A suc…"
"""EHOAPHDI_01174""",0.0,"""SRR5240729""","""s__Phascolarctobacterium_A suc…"
"""EHOAPHDI_01174""",0.0,"""SRR12795790""","""s__Phascolarctobacterium_A suc…"
…,…,…,…
"""EHOAPHDI_01174""",1.0,"""ERR1135331""","""s__Phascolarctobacterium_A suc…"
"""EHOAPHDI_01174""",1.0,"""SRR11124923""","""s__Phascolarctobacterium_A suc…"
"""EHOAPHDI_01174""",1.0,"""SRR11489769""","""s__Phascolarctobacterium_A suc…"


In [9]:
depth_df.filter((pl.col("gene") == 'EHOAPHDI_01174')).sort(by='breadth').filter(pl.col("metag") == "ERR1135199")

gene,breadth,metag,species
str,f64,str,str
"""EHOAPHDI_01174""",0.279642,"""ERR1135199""","""s__Phascolarctobacterium_A suc…"
